# 項目：用線性迴歸預測房價數據

## 分析目標

此數據分析報告的目的是，基於已有的房屋銷售價格，以及有關該房屋的屬性，進行線性迴歸分析，從而利用得到的線性迴歸模型，能對以下未知售價的房屋根據屬性進行價格預測：

面積為 6500 平方英尺，有 4 個臥室、2 個廁所，總共 2 層，不位於主路，無客房，帶地下室，有熱水器，沒有空調，車位數為 2，位於城市首選社區，簡裝修。

## 簡介

數據集 house_price.csv 記錄了超過五百棟房屋的交易價格，以及房屋的相關屬性資訊，包括房屋面積、臥室數、廁所數、樓層數、是否位於主路、是否有客房，等等。

`house_price.csv`每列的含意如下：
- price：房屋出售价格
- area：房屋面積，以平方英尺為單位
- bedrooms：卧室數
- bathrooms：廁所數
- stories：樓層數
- mainroad：是否位於主路
   - yes  是
   - no	  否
- guestroom：是否有客房
   - yes  是
   - no	  否
- basement：是否有地下室
   - yes  是
   - no	  否
- hotwaterheating：是否有熱水器
   - yes  是
   - no	  否
- airconditioning：是否有空調
   - yes  是
   - no	  否
- parking：車庫容量，以車輛數量為單位
- prefarea：是否位于城市首選社區
   - yes  是
   - no	  否
- furnishingstatus：装修狀態
   - furnished       精装
   - semi-furnished	 簡裝
   - unfurnished     毛坯

In [27]:
import numpy as np
import pandas as pd

In [3]:
original_data = pd.read_csv(r"C:\Users\USER\Desktop\house_price.csv")

簡單觀察數據，看有無需要清理

In [5]:
original_data.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [6]:
original_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [7]:
cleaned_data = original_data.copy()

In [8]:
cleaned_data['mainroad'] = cleaned_data['mainroad'].astype("category")
cleaned_data['guestroom'] = cleaned_data['guestroom'].astype("category")
cleaned_data['basement'] = cleaned_data['basement'].astype("category")
cleaned_data['hotwaterheating'] = cleaned_data['hotwaterheating'].astype("category")
cleaned_data['airconditioning'] = cleaned_data['airconditioning'].astype("category")
cleaned_data['prefarea'] = cleaned_data['prefarea'].astype("category")
cleaned_data['furnishingstatus'] = cleaned_data['furnishingstatus'].astype("category")

In [ ]:
數據基本無缺失值，惟mainroad,guestroom,basement,hotwaterheating,airconditioning,prefarea,furninshingstatus都是分類數據，
因此可將數據型態轉為category，接著檢查這些數據不一致，亦即這些數據是否有不同值實際指代同一目標的情況。

In [9]:
cleaned_data['mainroad'].value_counts()

yes    468
no      77
Name: mainroad, dtype: int64

In [10]:
cleaned_data['guestroom'].value_counts()

no     448
yes     97
Name: guestroom, dtype: int64

In [11]:
cleaned_data['basement'].value_counts()

no     354
yes    191
Name: basement, dtype: int64

In [12]:
cleaned_data['hotwaterheating'].value_counts()

no     520
yes     25
Name: hotwaterheating, dtype: int64

In [13]:
cleaned_data['airconditioning'].value_counts()

no     373
yes    172
Name: airconditioning, dtype: int64

In [14]:
cleaned_data['prefarea'].value_counts()

no     417
yes    128
Name: prefarea, dtype: int64

In [15]:
cleaned_data['furnishingstatus'].value_counts()

semi-furnished    227
unfurnished       178
furnished         140
Name: furnishingstatus, dtype: int64

從以上輸出結果來看，均不存在數據不一致的情況。

In [16]:
cleaned_data.describe()

,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


In [ ]:
從以上统計信息來看，cleaned_house_price裡不存在脫離現實意義的值。

In [28]:
import statsmodels.api as sm
lr_house_price = cleaned_data.copy()
lr_house_price

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [18]:
#將分類數據引入虛擬變數
lr_house_price = pd.get_dummies(lr_house_price, drop_first = True , columns=['mainroad', 'guestroom',
                                                                             'basement', 'hotwaterheating',
                                                                             'airconditioning','prefarea', 
                                                                             'furnishingstatus'],dtype = int)
lr_house_price

,price,area,bedrooms,bathrooms,stories,parking,mainroad_yes,guestroom_yes,basement_yes,hotwaterheating_yes,airconditioning_yes,prefarea_yes,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,13300000,7420,4,2,3,2,1,0,0,0,1,1,0,0
1,12250000,8960,4,4,4,3,1,0,0,0,1,0,0,0
2,12250000,9960,3,2,2,2,1,0,1,0,0,1,1,0
3,12215000,7500,4,2,2,3,1,0,1,0,1,1,0,0
4,11410000,7420,4,1,2,2,1,1,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,2,1,0,1,0,0,0,0,1
541,1767150,2400,3,1,1,0,0,0,0,0,0,0,1,0
542,1750000,3620,2,1,1,0,1,0,0,0,0,0,0,1
543,1750000,2910,3,1,1,0,0,0,0,0,0,0,0,0


In [19]:
#定義自變數以及應變數
y = lr_house_price['price']
x = lr_house_price.drop('price',axis = 1 )

In [22]:
#檢查自變數是否存在共線性問題(以相關性大於0.8為依據)
x.corr().abs()>0.8

,area,bedrooms,bathrooms,stories,parking,mainroad_yes,guestroom_yes,basement_yes,hotwaterheating_yes,airconditioning_yes,prefarea_yes,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
area,True,False,False,False,False,False,False,False,False,False,False,False,False
bedrooms,False,True,False,False,False,False,False,False,False,False,False,False,False
bathrooms,False,False,True,False,False,False,False,False,False,False,False,False,False
stories,False,False,False,True,False,False,False,False,False,False,False,False,False
parking,False,False,False,False,True,False,False,False,False,False,False,False,False
mainroad_yes,False,False,False,False,False,True,False,False,False,False,False,False,False
guestroom_yes,False,False,False,False,False,False,True,False,False,False,False,False,False
basement_yes,False,False,False,False,False,False,False,True,False,False,False,False,False
hotwaterheating_yes,False,False,False,False,False,False,False,False,True,False,False,False,False
airconditioning_yes,False,False,False,False,False,False,False,False,False,True,False,False,False


從上述結果來看，各項自變數除了自己以外對其他變數相關性均小於0.8，因此不存在共線性問題

In [34]:
#給迴歸模型添加截距
x = sm.add_constant(x)
x

,const,area,bedrooms,bathrooms,stories,parking,mainroad_yes,guestroom_yes,basement_yes,hotwaterheating_yes,airconditioning_yes,prefarea_yes,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,1.0,7420,4,2,3,2,1,0,0,0,1,1,0,0
1,1.0,8960,4,4,4,3,1,0,0,0,1,0,0,0
2,1.0,9960,3,2,2,2,1,0,1,0,0,1,1,0
3,1.0,7500,4,2,2,3,1,0,1,0,1,1,0,0
4,1.0,7420,4,1,2,2,1,1,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1.0,3000,2,1,1,2,1,0,1,0,0,0,0,1
541,1.0,2400,3,1,1,0,0,0,0,0,0,0,1,0
542,1.0,3620,2,1,1,0,1,0,0,0,0,0,0,1
543,1.0,2910,3,1,1,0,0,0,0,0,0,0,0,0


In [37]:
#分析預測
model = sm.OLS(y,x).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  price   R-squared:                       0.682
Model:                            OLS   Adj. R-squared:                  0.674
Method:                 Least Squares   F-statistic:                     87.52
Date:                Fri, 07 Nov 2025   Prob (F-statistic):          9.07e-123
Time:                        14:29:01   Log-Likelihood:                -8331.5
No. Observations:                 545   AIC:                         1.669e+04
Df Residuals:                     531   BIC:                         1.675e+04
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
===================================================================================================
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const                            4.277e+04   2.64e+05      0.162      0.872   -4.76e+05    5.62e+05
area                              244.1394     24.289     10.052      0.000     196.425     291.853
bedrooms                         1.148e+05   7.26e+04      1.581      0.114   -2.78e+04    2.57e+05
bathrooms                        9.877e+05   1.03e+05      9.555      0.000    7.85e+05    1.19e+06
stories                          4.508e+05   6.42e+04      7.026      0.000    3.25e+05    5.77e+05
parking                          2.771e+05   5.85e+04      4.735      0.000    1.62e+05    3.92e+05
mainroad_yes                     4.213e+05   1.42e+05      2.962      0.003    1.42e+05    7.01e+05
guestroom_yes                    3.005e+05   1.32e+05      2.282      0.023    4.18e+04    5.59e+05
basement_yes                     3.501e+05    1.1e+05      3.175      0.002    1.33e+05    5.67e+05
hotwaterheating_yes              8.554e+05   2.23e+05      3.833      0.000    4.17e+05    1.29e+06
airconditioning_yes               8.65e+05   1.08e+05      7.983      0.000    6.52e+05    1.08e+06
prefarea_yes                     6.515e+05   1.16e+05      5.632      0.000    4.24e+05    8.79e+05
furnishingstatus_semi-furnished -4.634e+04   1.17e+05     -0.398      0.691   -2.75e+05    1.83e+05
furnishingstatus_unfurnished    -4.112e+05   1.26e+05     -3.258      0.001   -6.59e+05   -1.63e+05
==============================================================================
Omnibus:                       97.909   Durbin-Watson:                   1.209
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              258.281
Skew:                           0.895   Prob(JB):                     8.22e-57
Kurtosis:                       5.859   Cond. No.                     3.49e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.49e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

當顯著水準為0.05時，臥室數量以及是否為簡裝房對於房屋價格的預測沒有顯著的預測作用。另外，常數的P值也超過顯著水準，表系統認為常數不是一個非0的數字。
接著會把這些變量移除後再次測試迴歸模型。

In [38]:
x = x.drop(['const','bedrooms','furnishingstatus_semi-furnished'],axis =1)

In [40]:
model = sm.OLS(y,x).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:                  price   R-squared (uncentered):                   0.957
Model:                            OLS   Adj. R-squared (uncentered):              0.956
Method:                 Least Squares   F-statistic:                              1088.
Date:                Fri, 07 Nov 2025   Prob (F-statistic):                        0.00
Time:                        14:36:50   Log-Likelihood:                         -8333.5
No. Observations:                 545   AIC:                                  1.669e+04
Df Residuals:                     534   BIC:                                  1.674e+04
Df Model:                          11                                                  
Covariance Type:            nonrobust                                                  
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
area                           257.5854     22.577     11.409      0.000     213.234     301.936
bathrooms                     1.071e+06   9.27e+04     11.553      0.000    8.89e+05    1.25e+06
stories                       5.084e+05   5.71e+04      8.898      0.000    3.96e+05    6.21e+05
parking                       2.793e+05   5.83e+04      4.794      0.000    1.65e+05    3.94e+05
mainroad_yes                  4.672e+05   1.27e+05      3.679      0.000    2.18e+05    7.17e+05
guestroom_yes                 2.851e+05   1.31e+05      2.172      0.030    2.72e+04    5.43e+05
basement_yes                  4.016e+05   1.07e+05      3.765      0.000    1.92e+05    6.11e+05
hotwaterheating_yes           8.668e+05   2.23e+05      3.884      0.000    4.28e+05    1.31e+06
airconditioning_yes           8.543e+05   1.07e+05      7.952      0.000    6.43e+05    1.07e+06
prefarea_yes                  6.443e+05   1.15e+05      5.594      0.000    4.18e+05    8.71e+05
furnishingstatus_unfurnished -3.493e+05   9.49e+04     -3.679      0.000   -5.36e+05   -1.63e+05
==============================================================================
Omnibus:                       94.840   Durbin-Watson:                   1.262
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              251.889
Skew:                           0.865   Prob(JB):                     2.01e-55
Kurtosis:                       5.845   Cond. No.                     2.74e+04
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[3] The condition number is large, 2.74e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

把上述變數剔除後，可發現模型的解釋力(R-square)由0.682提升到0.957。此外從係數觀察可發現除了毛坯房這個數據與房價是負相關以外，其餘變數皆為正相關。亦即如果房屋是毛坯房的話會使房屋價格降低。

In [63]:
# 要預測房價的房屋信息：
# 面積6500平方英尺，有4個卧室、2個廁所，總共2層，不位於主路，無客人房，有地下室，有熱水器，没有空調，車位數2，位於城市首選社區，簡装修

In [71]:
prediction = pd.DataFrame({'area':['6500'],'bedrooms':['4'],'bathrooms':['2'],
                           'stories':['2'],'mainroad':['no'],'guestroom': ['no'],
                           'basement': ['yes'], 'hotwaterheating': ['yes'],
                           'airconditioning': ['no'], 'parking': 2, 'prefarea': ['yes'],
                           'furnishingstatus':['semi-furnished']
                          })
prediction

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,6500,4,2,2,no,no,yes,yes,no,2,yes,semi-furnished


In [72]:
prediction['mainroad'] = pd.Categorical(prediction['mainroad'],categories = ['no','yes'])
prediction['guestroom'] = pd.Categorical(prediction['guestroom'],categories = ['yes','no'])
prediction['basement'] = pd.Categorical(prediction['basement'],categories = ['yes','no'])
prediction['hotwaterheating'] = pd.Categorical(prediction['hotwaterheating'],categories = ['yes','no'])
prediction['airconditioning'] = pd.Categorical(prediction['airconditioning'],categories = ['yes','no'])
prediction['prefarea'] = pd.Categorical(prediction['prefarea'],categories = ['yes','no'])
prediction['furnishingstatus'] = pd.Categorical(prediction['furnishingstatus'],categories = ['furnished', 'semi-furnished', 'unfurnished'])

In [73]:
prediction

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,6500,4,2,2,no,no,yes,yes,no,2,yes,semi-furnished


In [74]:
prediction = pd.get_dummies(prediction, drop_first = True,
                           columns = ['mainroad','guestroom','basement',
                                      'hotwaterheating','airconditioning','prefarea',
                                      'furnishingstatus'],dtype = int)
prediction

,area,bedrooms,bathrooms,stories,parking,mainroad_yes,guestroom_no,basement_no,hotwaterheating_no,airconditioning_no,prefarea_no,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,6500,4,2,2,2,0,1,0,0,1,0,1,0


In [77]:
#剔除掉之前無顯著相關的兩個變數
prediction = prediction.drop(['bedrooms','furnishingstatus_semi-furnished'],axis =1)

In [83]:
#運行時發現有變數尚未轉化為數值因此出錯，先將area, bathrooms以及stories轉化為整數
prediction['area'] = prediction['area'].astype(int)
prediction['bathrooms'] = prediction['bathrooms'].astype(int)
prediction['stories'] = prediction['stories'].astype(int)

In [84]:
predicted_value = model.predict(prediction)
predicted_value

0    6.530508e+06
dtype: float64

預測價格6,530,508